# US equities panel: from a ranking to a book, at equal weight

Everything before this notebook produced **predictions**: a number per stock per session saying
how that stock is expected to rank. Nothing has yet held a position. This is where a ranking
becomes a portfolio, and it is the first point in the case study where the answer can be
measured in money rather than in correlation.

**The rule is deliberately the plainest one available.** On each rebalancing date, sort the
stocks by the model's prediction, go long the top *k* and short the bottom *k*, and put the same
amount of money in every position. No weighting by conviction, no risk targeting, no overlay -
those come in the two notebooks after this one, and each is measured against what this produces.

**Why equal weight is the right baseline rather than a weak one.** A sizing rule can flatter a
ranking or bury it, so measuring a model under a clever allocator confounds two things. Equal
weight adds the least: whatever separates two models here is the ranking, because nothing else
differs. It is also the hardest baseline to beat by accident, which is what makes the comparison
in [`17_portfolio_management`](17_portfolio_management.ipynb) worth making.

**Every member of every model population is backtested, not a shortlist.** A model that ranked
poorly on information coefficient is still run, because ranking accuracy and strategy
performance are different questions - a model can order the cross-section well and trade so much
that turnover consumes the whole of it, or rank indifferently and hold a book that does not.
Selecting on the ranking measure before backtesting would decide the second question with the
answer to the first.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國股票面板：以等權重將排序轉為持倉

本 Notebook 之前產生的都是**預測**：每檔股票每個交易時段一個數值，表示預期排名。先前尚未建立部位。此處將排序轉成投資組合，也是案例研究首次能以金錢而非相關性衡量答案的地方。

**規則刻意採用最簡單的版本。** 每個再平衡日期依模型預測排序股票，買進前 *k* 檔、放空後 *k* 檔，每個部位投入相同金額。不依信心程度加權、不設定風險目標、也不加覆蓋策略；這些內容由後面兩份 Notebook 處理，並各自與此處結果比較。

**等權重是適當基準，而非薄弱基準。** 部位大小規則可能美化或掩蓋排序效果，因此若在複雜配置器下量測模型，就會混淆兩種因素。等權重加入的影響最少：此處兩個模型的差異來自排序，因為其他條件相同。它也最不容易被偶然擊敗，因此 [`17_portfolio_management`](17_portfolio_management.ipynb) 的比較才值得進行。

**每個模型群組的所有成員都會回測，而非只選短名單。** 即使某模型的資訊係數排序表現不佳，仍會執行，因為排序準確度與策略績效是不同問題。模型可能善於排列橫斷面，卻交易過度，使換手耗盡收益；也可能排序平平，卻持有有效的部位組合。若在回測前按排序指標篩選，就是拿第一個問題的答案決定第二個問題。
<!-- ML4T_ZH_END c0000-p000 -->

**This is where selection genuinely begins.** Every notebook so far has said "selection happens
in `16_backtest`". This notebook produces the validation backtests it happens on; the choice
itself is made in the strategy notebooks that follow, over the population frozen here.

**Learning objectives.** By the end of this notebook you will be able to:

- Describe how a cross-sectional ranking becomes a long-short book, and name every decision a
  rebalance has to make.
- Say why the plainest sizing rule is the right one for a baseline, and what a comparison under
  a sophisticated allocator would confound.
- Say why the whole model population is backtested rather than a shortlist chosen on ranking
  accuracy.
- Read a validation Sharpe across a grid of models and top-*k* choices and say what varies along
  each axis.

**Book reference**: Chapter 16, Sections 16.4 to 16.8.

**Prerequisites**: the model notebooks - [`06_linear`](06_linear.ipynb) through
[`13b_ipca`](13b_ipca.ipynb) - have frozen the named prediction sets listed below, and
[`15_model_analysis`](15_model_analysis.ipynb) has confirmed each is complete.

**What it writes**: one validation backtest per prediction set member and top-*k* choice, in
`run_log/registry.db`, frozen as one named baseline set per label.
[`17_portfolio_management`](17_portfolio_management.ipynb) varies the sizing on what wins here,
[`18_risk_management`](18_risk_management.ipynb) lays overlays on top, and
[`19_costs`](19_costs.ipynb) charges the result for trading.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**選擇真正從此處開始。** 之前每份 Notebook 都說「選擇在 `16_backtest` 進行」。本 Notebook 產生作為選擇依據的驗證回測；實際選擇由後續策略 Notebook 在此處固定的模型群組上進行。

**學習目標。** 完成本 Notebook 後，你將能夠：

- 描述橫斷面排序如何轉成多空部位組合，並指出每次再平衡須做的各項決策。
- 說明最簡單的部位大小規則為何適合作為基準，以及在複雜配置器下比較會混淆什麼。
- 說明為何回測整個模型群組，而非依排序準確度挑選短名單。
- 解讀模型與前 *k* 檔選擇所形成的驗證夏普比率網格，並說明各軸變動的是什麼。

**書籍參考**：第 16 章第 16.4 至 16.8 節。

**先決條件**：模型 Notebook，從 [`06_linear`](06_linear.ipynb) 到 [`13b_ipca`](13b_ipca.ipynb)，已固定下列具名預測集合，且 [`15_model_analysis`](15_model_analysis.ipynb) 已確認各集合完整。

**產出內容**：對每位預測集合成員及每個前 *k* 檔選擇，各寫出一項驗證回測，存於 `run_log/registry.db`，並依標籤固定為具名基準集合。[`17_portfolio_management`](17_portfolio_management.ipynb) 調整勝出模型的部位大小，[`18_risk_management`](18_risk_management.ipynb) 加上覆蓋策略，[`19_costs`](19_costs.ipynb) 則計入交易成本。
<!-- ML4T_ZH_END c0000-p001 -->


In [ ]:
"""Generate the complete US-equities equal-weight validation baseline."""

import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from case_studies.research import (
    CandidateSet,
    OfficialPopulation,
    candidate_set_supersedes,
    open_study,
    plan_backtests,
    population_supersedes,
    run_backtests,
)
from case_studies.utils.backtest_loaders import get_backtest_config, load_backtest_prices_for
from case_studies.utils.sweep_config import get_entry_schemes_for
from utils.style import COLORS, add_message_title, show_with_alt, zero_line

### Which prediction sets enter the pool

Every model notebook froze its results under a name per label and family, and the list below is
the pool this baseline ranks. A name missing from it is a family that never competes, so the
list is the declaration of what the strategy chain is allowed to choose from.

**The weekly experiment is deliberately absent.** [`12_dl_weekly`](12_dl_weekly.ipynb) fits
`fwd_ret_5d` on a Friday-only grid, and its predictions carry the same label as the daily models
below. Ranking them together would compare series scored on different sets of decision dates -
one on every session, one on about a fifth of them - and read the difference as a difference
between models. It runs and registers; it is not a candidate here.

<!-- ML4T_ZH_BEGIN c0002-p000 -->
### 哪些預測集合進入候選池

每份模型 Notebook 都依標籤和類別將結果固定於一個名稱之下；下列清單就是此基準用來排序的候選池。若某個名稱缺席，該類別就永遠不參與競爭，因此清單宣告策略流程可從哪些項目選擇。

**刻意排除週頻實驗。** [`12_dl_weekly`](12_dl_weekly.ipynb) 在僅星期五的網格上擬合 `fwd_ret_5d`，其預測與下方每日模型使用相同標籤。若一起排序，就會比較在不同決策日期集合上評分的序列：一方每個交易時段都有，另一方只有約五分之一，並將差異誤讀為模型差異。該實驗會執行與登錄，但不是此處候選項。
<!-- ML4T_ZH_END c0002-p000 -->


In [ ]:
CASE_STUDY_ID = "us_equities_panel"
PREDICTION_SET_NAMES = [
    "us-equities-fwd-ret-1d-linear-v1",
    "us-equities-fwd-ret-5d-linear-v1",
    "us-equities-fwd-ret-21d-linear-v1",
    "us-equities-fwd-ret-1d-gbm-v1",
    "us-equities-fwd-ret-5d-gbm-v1",
    "us-equities-fwd-ret-21d-gbm-v1",
    "us-equities-fwd-ret-1d-tabular-dl-v1",
    "us-equities-fwd-ret-1d-nlinear-v1",
    "us-equities-fwd-ret-1d-lstm-v1",
    "us-equities-fwd-ret-1d-pca-v1",
    "us-equities-fwd-ret-1d-ipca-v1",
]
EXECUTION_TIER = "canonical"
POPULATION_NAME = ""
SUPERSEDES_POPULATION = ""
SUPERSEDES_SETS: dict = {}
# Empty means this run writes to the case study's own store, which is what canonical
# production execution wants. Any other value routes the run's writes there instead, at
# either tier, and is how a rehearsal at full scale is compared against the published
# result without being able to damage it.
WORKSPACE = ""
PREVIEW_LABELS = []
PREVIEW_FAMILIES = []
PREVIEW_CONFIG_NAMES = []
PREVIEW_MAX_PREDICTIONS = 0
MAX_SYMBOLS = 0

## 2. Every model, no shortlist

The named sets above are opened and their membership checked, and every member goes on to be
backtested. Both tiers resolve the study through `open_study`, which reads the labels and
features in place and redirects only writes, so a preview run scores the same inputs a canonical
one does and cannot publish over it.

Nothing here filters on a predictive metric, for the reason the preamble gives: a
model that ranks the cross-section well can still trade too much for anything to be left after
turnover, and one that ranks indifferently can hold a book that works. Deciding the second
question with the answer to the first is the mistake the whole design avoids.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 2. 每個模型都納入，沒有短名單

開啟上述具名集合、檢查其成員，並逐一回測。兩種執行層級都透過 `open_study` 解析研究；它就地讀取標籤與特徵，只重新導向寫入，因此預覽執行使用與標準執行相同的輸入，卻無法覆蓋標準結果。

此處不根據預測指標篩選，理由見前言：善於排序橫斷面的模型，可能交易過度，使換手後所剩無幾；排序平平的模型，也可能持有有效的部位組合。拿第一個問題的答案決定第二個問題，正是整體設計要避免的錯誤。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
preview_filters = bool(PREVIEW_LABELS or PREVIEW_FAMILIES or PREVIEW_CONFIG_NAMES)
workspace_override = os.environ.get("ML4T_OUTPUT_DIR") or WORKSPACE
if EXECUTION_TIER == "canonical":
    if preview_filters or PREVIEW_MAX_PREDICTIONS or MAX_SYMBOLS:
        raise ValueError("Canonical execution cannot declare preview reductions")
    if not PREDICTION_SET_NAMES or len(PREDICTION_SET_NAMES) != len(set(PREDICTION_SET_NAMES)):
        raise ValueError("Canonical execution requires unique named prediction sets")
    study = open_study(
        CASE_STUDY_ID,
        execution_tier=EXECUTION_TIER,
        workspace=Path(workspace_override) if workspace_override else None,
    )
elif EXECUTION_TIER == "preview":
    if not preview_filters or PREVIEW_MAX_PREDICTIONS < 1 or MAX_SYMBOLS < 1:
        raise ValueError(
            "Preview execution requires a catalog filter, prediction limit, and symbol limit"
        )
    study = open_study(
        CASE_STUDY_ID,
        execution_tier=EXECUTION_TIER,
        workspace=Path(workspace_override or "experiments"),
    )
else:
    raise ValueError(f"Unsupported execution tier: {EXECUTION_TIER!r}")

## 3. Which rows can be backtested at all

A prediction row is eligible when it is complete, scored on the validation split, and produced
under this run's execution tier. A row failing any of those is not a weaker candidate; it is a
row that would make the population mean something other than what it says, so the run refuses
rather than dropping it silently.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 3. 哪些資料列可以回測

預測資料列必須完整、在驗證分割上評分，且由本次執行層級產生，才符合資格。不符合任何一項的資料列，並非較弱的候選項，而是會讓模型群組意義偏離宣告的資料列；因此執行會拒絕它，而非默默捨棄。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
prediction_catalog = study.predictions.table(include_preview=True)
if EXECUTION_TIER == "canonical":
    prediction_sets = tuple(CandidateSet.one(study, name=name) for name in PREDICTION_SET_NAMES)
    if any(result_set.member_kind != "prediction" for result_set in prediction_sets):
        raise ValueError("Every declared input set must contain predictions")
    prediction_members = tuple(
        member for result_set in prediction_sets for member in result_set.members
    )
    if len(prediction_members) != len(set(prediction_members)):
        raise ValueError("Declared prediction sets overlap")
    predictions = prediction_catalog.filter(pl.col("prediction_hash").is_in(prediction_members))
    if predictions.height != len(prediction_members):
        raise ValueError("The prediction catalog does not contain every declared member")
else:
    predictions = prediction_catalog.filter(pl.col("execution_tier") == "preview")
    if PREVIEW_LABELS:
        predictions = predictions.filter(pl.col("label").is_in(PREVIEW_LABELS))
    if PREVIEW_FAMILIES:
        predictions = predictions.filter(pl.col("family").is_in(PREVIEW_FAMILIES))
    if PREVIEW_CONFIG_NAMES:
        predictions = predictions.filter(pl.col("config_name").is_in(PREVIEW_CONFIG_NAMES))
    predictions = predictions.sort("prediction_hash").head(PREVIEW_MAX_PREDICTIONS)

ineligible = predictions.filter(
    (pl.col("split") != "validation")
    | (pl.col("execution_tier") != EXECUTION_TIER)
    | ~pl.col("complete")
)
if predictions.is_empty() or not ineligible.is_empty():
    raise ValueError(
        "The baseline requires complete validation predictions from one execution tier"
    )

The selected rows expose the model family, configuration, label, checkpoint, and exact result
identities that enter the baseline.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
選取的資料列呈現進入基準的模型類別、設定、標籤、檢查點，以及精確的結果識別資訊。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
prediction_population = predictions.select(
    "family",
    "config_name",
    "label",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
).sort("label", "family", "config_name", "checkpoint_value", "prediction_hash")
prediction_population

## 4. Planning the backtests before running any

Each prediction row crossed with each declared top-*k* becomes one planned backtest, and every
identity is written down before the first one runs. That is what makes a run that comes up short
read as a failure rather than as a smaller experiment, and it is what lets an interrupted run
resume by filling what is missing.

**The top-*k* grid is a strategy decision, not a tuning knob.** Holding 20 names a side and
holding 50 are different strategies with different concentration and different turnover, and
both are backtested rather than one being chosen in advance.#
`SUPERSEDES_POPULATION` and `SUPERSEDES_SETS` name the generation this run replaces. A population
and a candidate set are both immutable, so a re-run that admits different members has to say
which snapshot it supersedes or the registry refuses the write. Both default to empty, which is
right for a first run and for a reader's clean clone; `population_supersedes` and
`candidate_set_supersedes` withhold a declared hash wherever offering it would be refused.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 4. 執行前先規劃回測

每列預測與每個已宣告的前 *k* 檔選擇交叉組合，形成一項計畫回測；第一項回測開始前，所有識別資訊都已記錄。這使執行不足被視為失敗，而非較小的實驗，也讓中斷的執行能補齊缺口後繼續。

**前 *k* 檔網格是策略決策，不是調參旋鈕。** 每邊持有 20 檔與 50 檔，是集中度和換手率都不同的策略；兩者都會回測，而非事先選定其一。#
`SUPERSEDES_POPULATION` 與 `SUPERSEDES_SETS` 指定本次執行要取代的世代。模型群組與候選集合都不可變，因此若重新執行納入不同成員，必須說明取代哪個快照，否則登錄系統會拒絕寫入。兩者預設留空，適用於首次執行或讀者剛取得的乾淨副本；`population_supersedes` 與 `candidate_set_supersedes` 會在提供已宣告雜湊值將遭拒絕時省略它。
<!-- ML4T_ZH_END c0010-p000 -->


In [ ]:
backtest_config = get_backtest_config(CASE_STUDY_ID)
planned_requests = []
plan_rows = []


def plan_baseline_member(label, prices, scheme, prediction_row):
    selected_prediction = predictions.filter(
        pl.col("prediction_hash") == prediction_row["prediction_hash"]
    )
    signal = {key: value for key, value in scheme.items() if key != "name"}
    plan = plan_backtests(
        study,
        predictions=selected_prediction,
        signal=signal,
        prices=prices,
        chapter="ch16",
    )
    if len(plan.members) != 1:
        raise RuntimeError("One prediction and signal request must plan one backtest")
    expected_hash = plan.expected_hashes[0]
    request = {
        "label": label,
        "prediction_hash": prediction_row["prediction_hash"],
        "selection": selected_prediction,
        "signal_name": scheme["name"],
        "signal": signal,
        "expected_hash": expected_hash,
    }
    row = {
        "label": label,
        "family": prediction_row["family"],
        "config_name": prediction_row["config_name"],
        "checkpoint_kind": prediction_row["checkpoint_kind"],
        "checkpoint_value": prediction_row["checkpoint_value"],
        "signal": scheme["name"],
        "prediction_hash": prediction_row["prediction_hash"],
        "backtest_hash": expected_hash,
    }
    return request, row

In [ ]:
for label in predictions.get_column("label").unique().sort().to_list():
    label_predictions = predictions.filter(pl.col("label") == label)
    prices = load_backtest_prices_for(
        CASE_STUDY_ID,
        label,
        split="validation",
        max_symbols=MAX_SYMBOLS,
    )
    n_assets = prices.get_column("symbol").n_unique()
    schemes = get_entry_schemes_for(
        CASE_STUDY_ID,
        label,
        n_assets,
        long_short=backtest_config.long_short,
    )
    if not schemes or any(scheme["method"] != "equal_weight_top_k" for scheme in schemes):
        raise ValueError(f"{label} does not declare only equal-weight baseline schemes")
    for prediction_row in label_predictions.iter_rows(named=True):
        for scheme in schemes:
            request, row = plan_baseline_member(label, prices, scheme, prediction_row)
            planned_requests.append(request)
            plan_rows.append(row)
    del prices

In [ ]:
planned_population = pl.DataFrame(plan_rows).sort(
    "label", "family", "config_name", "checkpoint_value", "signal", "backtest_hash"
)
if planned_population.get_column("backtest_hash").n_unique() != planned_population.height:
    raise ValueError("The baseline plan contains duplicate backtest identities")

official_population = None
if EXECUTION_TIER == "canonical":
    population_name = POPULATION_NAME or "us-equities-baseline-v1"
    official_population = OfficialPopulation.create(
        study,
        name=population_name,
        supersedes=population_supersedes(
            study, name=population_name, declared=SUPERSEDES_POPULATION
        ),
        member_kind="backtest",
        members=tuple(planned_population.get_column("backtest_hash")),
    )

planned_population

## 5. Running them

Each planned backtest is independent, so a failure costs that member and leaves the rest usable,
and the declared membership stays intact for a re-run to fill.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 5. 執行回測

每項計畫回測都獨立，因此失敗只影響該成員，其他結果仍可使用；宣告的成員清單保持不變，重新執行可補齊缺口。
<!-- ML4T_ZH_END c0014-p000 -->


In [ ]:
execution_rows = []
failure_rows = []


def execute_baseline_member(prices, request):
    execution = run_backtests(
        study,
        predictions=request["selection"],
        signal=request["signal"],
        prices=prices,
        chapter="ch16",
    )
    if len(execution.results) != 1 or execution.results[0].hash != request["expected_hash"]:
        raise RuntimeError("Baseline execution changed its planned identity")
    return {
        "label": request["label"],
        "prediction_hash": request["prediction_hash"],
        "signal": request["signal_name"],
        "backtest_hash": execution.results[0].hash,
        "status": execution.diagnostics[0]["status"],
    }

In [ ]:
for label in predictions.get_column("label").unique().sort().to_list():
    prices = load_backtest_prices_for(
        CASE_STUDY_ID,
        label,
        split="validation",
        max_symbols=MAX_SYMBOLS,
    )
    for request in (item for item in planned_requests if item["label"] == label):
        try:
            execution_rows.append(execute_baseline_member(prices, request))
        except Exception as error:
            failure_rows.append(
                {
                    "label": label,
                    "prediction_hash": request["prediction_hash"],
                    "signal": request["signal_name"],
                    "backtest_hash": request["expected_hash"],
                    "error_type": type(error).__name__,
                    "error": str(error),
                }
            )
    del prices

In [ ]:
execution_diagnostics = pl.DataFrame(
    execution_rows,
    schema={
        "label": pl.String,
        "prediction_hash": pl.String,
        "signal": pl.String,
        "backtest_hash": pl.String,
        "status": pl.String,
    },
)
failures = pl.DataFrame(
    failure_rows,
    schema={
        "label": pl.String,
        "prediction_hash": pl.String,
        "signal": pl.String,
        "backtest_hash": pl.String,
        "error_type": pl.String,
        "error": pl.String,
    },
)
if not failures.is_empty():
    raise RuntimeError(f"Baseline population has {failures.height} unsuccessful members")

if official_population is not None:
    official_population.require_complete()

execution_diagnostics

## 6. Naming the baseline sets

One frozen set per label, under a name the later notebooks open by. Only an unnarrowed canonical
run publishes one, because a name must not mean two different member sets at two different times.

**The freeze is also the comparability check.** Nothing is declared comparable, so
`CandidateSet.create` requires every field of the protocol to be identical across the members:
two rows that measured their Sharpe on different folds are not two rankings of one thing, and
this is what refuses to freeze them together.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
## 6. 為基準集合命名

每個標籤各有一個固定集合，使用後續 Notebook 可開啟的名稱。只有未縮減的標準執行會發布名稱，因為名稱不可在不同時間代表不同成員集合。

**固定集合也是可比較性檢查。** 此處未宣告任何結果可直接比較，因此 `CandidateSet.create` 要求成員之間的每個協定欄位都相同：若兩列在不同折上量測夏普比率，就不是同一事物的兩種排序；此檢查會拒絕將它們一起固定。
<!-- ML4T_ZH_END c0018-p000 -->


In [ ]:
set_rows = []
completed = study.backtests.table(include_preview=True).filter(
    pl.col("backtest_hash").is_in(planned_population.get_column("backtest_hash"))
)
if (
    completed.height != planned_population.height
    or completed.filter(~pl.col("complete")).height
    or completed.filter(pl.col("stage") != "signal").height
    or completed.filter(pl.col("execution_tier") != EXECUTION_TIER).height
    or completed.filter(pl.col("sharpe").is_null() | ~pl.col("sharpe").is_finite()).height
):
    raise RuntimeError("The baseline catalog is incomplete or mis-staged")
if EXECUTION_TIER == "canonical":
    for label in completed.get_column("label").unique().sort().to_list():
        label_name = label.replace("_", "-")
        result_set_name = f"us-equities-{label_name}-baseline-v1"
        result_set = study.backtests.freeze(
            completed.filter(pl.col("label") == label),
            name=result_set_name,
            supersedes=candidate_set_supersedes(
                study, name=result_set_name, declared=SUPERSEDES_SETS.get(result_set_name, "")
            ),
        )
        set_rows.append(
            {"label": label, "set_name": result_set.name, "members": len(result_set.members)}
        )

compatible_sets = pl.DataFrame(
    set_rows,
    schema={"label": pl.String, "set_name": pl.String, "members": pl.Int64},
)
compatible_sets

## 7. What came out

One row per model, checkpoint and top-*k*. Read across the two axes separately: down a model at
fixed top-*k* is what the ranking was worth, and across top-*k* at fixed model is what
concentration did to it.

**These are gross of trading costs.** Every number here is what the book earned before anything
was charged for turning it over, and on a three-thousand-name panel rebalanced daily that is a
large omission by design - [`19_costs`](19_costs.ipynb) is where it is put back.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
## 7. 產生的結果

每個模型、檢查點及前 *k* 檔選擇各有一列。應分別沿兩個軸解讀：固定前 *k* 檔時沿模型方向看，可知排序帶來的價值；固定模型時沿前 *k* 檔方向看，可知集中度的影響。

**這些數值都未扣除交易成本。** 此處每個數字都是部位組合在支付換手成本前的收益；對每日再平衡、涵蓋三千檔股票的面板而言，這是刻意保留的一大缺漏，將在 [`19_costs`](19_costs.ipynb) 補回。
<!-- ML4T_ZH_END c0020-p000 -->


In [ ]:
groups = completed.select("label", "family").unique().sort("label", "family").rows(named=True)
fig, ax = plt.subplots(figsize=(9, 5))
for position, group in enumerate(groups):
    values = completed.filter(
        (pl.col("label") == group["label"]) & (pl.col("family") == group["family"])
    ).get_column("sharpe")
    jitter = (values.arg_sort().to_numpy() % 9 - 4) / 40
    ax.scatter(
        position + jitter,
        values.to_numpy(),
        alpha=0.45,
        s=18,
        color=COLORS["blue"],
        edgecolors="none",
    )
zero_line(ax)
ax.set_xticks(
    range(len(groups)),
    [f"{group['label']}\n{group['family']}" for group in groups],
)
ax.set_xlim(-0.5, len(groups) - 0.5)
ax.set_ylabel("Validation Sharpe")
add_message_title(
    ax,
    "Validation Sharpe of the equal-weight backtests, by label and family",
    subtitle="One point per complete equal-weight validation backtest, gross of costs",
)
show_with_alt(
    fig,
    "A strip plot with one column per label and model family. Each point is one complete "
    "equal-weight validation backtest, placed at its Sharpe ratio on the vertical axis and "
    "spread horizontally within its column so overlapping points stay visible. A dashed "
    "horizontal reference line marks zero.",
)

## What to notice

**A ranking and a strategy are not the same thing, and this is where they separate.** Every model
was scored on how well it ordered the cross-section. What it earns depends on that order *and*
on how often the order changes, because every change is a trade. A model can rank well and turn
its book over so fast that turnover consumes what the ranking earned, and the previous notebooks
had no way to see it.

**Equal weight is what makes the comparison about the models.** Nothing here estimates anything
from the data beyond the predictions themselves, so a difference between two rows is a difference
between two rankings. Every sizing rule added later has to earn its place against that.

**Top-*k* is a strategy axis, not a tuning axis.** Twenty names a side and fifty are different
strategies - different concentration, different turnover, different capacity - and both are
backtested rather than one being chosen up front.

**Everything here is gross.** No commission, no spread, no borrow cost on the short leg. On a
daily-rebalanced long-short book across three thousand names that omission is large and is put
back in [`19_costs`](19_costs.ipynb); until then no number on this page is what anyone would have
received.

**Known limitations.** Validation folds have been read many times over by the time a case study
reaches this notebook, so a validation Sharpe is a selection statistic rather than an estimate of
future performance - the holdout, opened once at the end, is what speaks to that. The book is
rebalanced on a fixed schedule with no capacity constraint, so a position is taken at whatever
size the rule implies whether or not the stock trades enough to absorb it.


<!-- ML4T_ZH_BEGIN c0022-p000 -->
## 觀察重點

**排序與策略不是同一回事，此處開始看得出差別。** 每個模型先前都依橫斷面排序能力評分；實際收益取決於排序*以及*排序變化的頻率，因為每次變化都會產生交易。模型可能排序良好，但部位換手過快，耗盡排序所帶來的收益；先前的 Notebook 無從觀察。

**等權重使比較聚焦於模型。** 此處除了預測本身，不再從資料估計任何內容，因此兩列差異就是兩種排序的差異。後續加入的各種部位大小規則，都必須證明比這項基準更有價值。

**前 *k* 檔是策略軸，不是調參軸。** 每邊二十檔與五十檔是不同策略，集中度、換手率及容量均不同；兩者都回測，而非事先選擇一種。

**此處全部為未扣成本結果。** 沒有佣金、買賣價差，也沒有空頭部位的借券成本。對三千檔股票每日再平衡的多空部位組合而言，此缺漏很大，會在 [`19_costs`](19_costs.ipynb) 補入；在此之前，本頁數值都不是任何人實際能收到的收益。

**已知限制。** 案例研究進展至此時，驗證折已被反覆查看，因此驗證夏普比率是選擇用統計量，而非未來績效估計；最後只開啟一次的保留樣本，才能用來評估後者。部位組合依固定時程再平衡，沒有容量限制，因此規則要求多大部位就建立多大，無論股票交易量能否承受。
<!-- ML4T_ZH_END c0022-p000 -->

**Next**: [`17_portfolio_management`](17_portfolio_management.ipynb) keeps these names and
changes how the money is spread across them.

<!-- ML4T_ZH_BEGIN c0022-p001 -->
**下一步**：[`17_portfolio_management`](17_portfolio_management.ipynb) 保留這些股票名稱，改變資金在其間的分配方式。
<!-- ML4T_ZH_END c0022-p001 -->
